In [60]:
def rotr32(n, c):
    # Performs a cyclic right shift with removed bits appearing on the left
    return ((n >> c) | (n << (32 - c))) & 0xFFFFFFFF

def sha256(message):
    # K values given from the SHA-256 config file
    K = [
        0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
        0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
        0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
        0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
        0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
        0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
        0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
        0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2
    ]
    # Initial variable values
    h = [
        0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
        0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19
    ]

    # Padding
    msg = bytearray(message)
    orig_len_bits = len(msg) * 8
    msg.append(0x80)

    # Leave 64 bits for length-padding
    while (len(msg) * 8 + 64) % 512 != 0:
        msg.append(0x00)

    # Add length padding
    msg.extend(orig_len_bits.to_bytes(8, 'big'))

    # Blocks
    for i in range(0, len(msg), 64):
        chunk = msg[i:i+64]
        w = [0] * 64

        # First 16 elements of a block
        for t in range(16):
            w[t] = int.from_bytes(chunk[t*4:t*4+4], 'big')

        # Rest of 64 elements processed
        for t in range(16, 64):
            s0 = rotr32(w[t-15], 7) ^ rotr32(w[t-15], 18) ^ (w[t-15] >> 3)
            s1 = rotr32(w[t-2], 17) ^ rotr32(w[t-2], 19) ^ (w[t-2] >> 10)
            w[t] = (w[t-16] + s0 + w[t-7] + s1) & 0xFFFFFFFF

        # Init values
        a, b, c, d, e, f, g, current_h = h

        # 64 iterations of the hash-function
        for t in range(64):
            # Compute t1
            S1 = rotr32(e, 6) ^ rotr32(e, 11) ^ rotr32(e, 25)
            ch = (e & f) ^ (~e & g)
            t1 = (current_h + S1 + ch + K[t] + w[t]) & 0xFFFFFFFF

            # Compute t2
            S0 = rotr32(a, 2) ^ rotr32(a, 13) ^ rotr32(a, 22)
            maj = (a & b) ^ (a & c) ^ (b & c)
            t2 = (S0 + maj) & 0xFFFFFFFF

            # Shift values for next iteration + compute e and a
            current_h = g
            g = f
            f = e
            e = (d + t1) & 0xFFFFFFFF
            d = c
            c = b
            b = a
            a = (t1 + t2) & 0xFFFFFFFF
        # Get final values
        h[0] = (a + h[0]) & 0xFFFFFFFF
        h[1] = (b + h[1]) & 0xFFFFFFFF
        h[2] = (c + h[2]) & 0xFFFFFFFF
        h[3] = (d + h[3]) & 0xFFFFFFFF
        h[4] = (e + h[4]) & 0xFFFFFFFF
        h[5] = (f + h[5]) & 0xFFFFFFFF
        h[6] = (g + h[6]) & 0xFFFFFFFF
        h[7] = (current_h + h[7]) & 0xFFFFFFFF

    # Return hex elements in a string to ease the testing
    return "".join(f"{x:08x}" for x in h)

In [61]:
assert sha256(b"abc") == "ba7816bf 8f01cfea 414140de 5dae2223 b00361a3 96177a9c b410ff61 f20015ad".replace(" ", ""), 'Test Case for "abc" failed'

assert sha256(b"") == "e3b0c442 98fc1c14 9afbf4c8 996fb924 27ae41e4 649b934c a495991b 7852b855".replace(" ", ""), 'Test Case for "" failed'

assert sha256(b"abcdbcdecdefdefgefghfghighijhijkijkljklmklmnlmnomnopnopq") == "248d6a61 d20638b8 e5c02693 0c3e6039 a33ce459 64ff2167 f6ecedd4 19db06c1".replace(" ", ""), 'Test Case for "abcdbcdecdefdefgefghfghighijhijkijkljklmklmnlmnomnopnopq" failed'

assert sha256(b"abcdefghbcdefghicdefghijdefghijkefghijklfghijklmghijklmnhijklmnoijklmnopjklmnopqklmnopqrlmnopqrsmnopqrstnopqrstu") == "cf5b16a7 78af8380 036ce59e 7b049237 0b249b11 e8f07a51 afac4503 7afee9d1".replace(" ", ""), 'Test Case for "abcdefghbcdefghicdefghijdefghijkefghijklfghijklmghijklmnhijklmnoijklmnopjklmnopqklmnopqrlmnopqrsmnopqrstnopqrstu" failed'

assert sha256(b"a" * 1000000) == "cdc76e5c 9914fb92 81a1c7e2 84d73e67 f1809a48 a497200e 046d39cc c7112cd0".replace(" ", ""), 'Test Case for "a" * 1000000 failed'

assert sha256(b"abcdefghbcdefghicdefghijdefghijkefghijklfghijklmghijklmnhijklmno" * 16777216) == "50e72a0e 26442fe2 552dc393 8ac58658 228c0cbf b1d2ca87 2ae43526 6fcd055e".replace(" ", ""), 'Test Case for "abcdefghbcdefghicdefghijdefghijkefghijklfghijklmghijklmnhijklmno" * 16,777,216 failed'


print("Tests were successfull!!!")


Tests were successfull!!!


Цю ^^^ тєму не рань, останній тест на годину

In [73]:
import numpy as np
import hashlib
import os

def find_prefix(message = b"give my friend 2 bitcoins for a pizza"):

  length = 20
  n_zeros = 32//8
  counter = 0

  while True:
    prefix = counter.to_bytes(20, 'big')
    # prefix = os.urandom(length)
    counter +=1
    message_pref = prefix + message
    result = hashlib.sha256(message_pref).digest()


    if result[:n_zeros] == b'\x00' * (n_zeros):
      return prefix, counter, result

In [74]:
prefix, counter, res_str = find_prefix()
print(f"Found the prefix {prefix} with {counter} steps. The result of the Hash function is {res_str}")

Found the prefix b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x9b\xdf\x14J' with 2615088203 steps. The result of the Hash function is b'\x00\x00\x00\x00\x16o\xa5\x05\xce\x1a\xc5\xe7\xf3\xef\\\xb2$X\x99\xdc \x0fC$m9\x89\x82GS\xca\xe4'


In [ ]:
%%capture
!openssl genpkey -algorithm RSA -out server.key -pkeyopt rsa_keygen_bits:8192

In [ ]:
!openssl req -new -key server.key -out server.csr -subj "/C=UA/L=Kyiv/O=KSE/CN=www.crypto.kse.ua emailAddress=aboichuk@kse.org.ua" -nodes

In [ ]:
!openssl req -text -noout -in server.csr

In [ ]:
!openssl x509 -req -sha256 -in server.csr -signkey server.key -out server.crt

In [ ]:
!openssl x509 -text -noout -in server.crt